<a href="https://colab.research.google.com/github/Disskaf/pib-colombia-arima/blob/main/pib_colombia_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Taller Final: Programación aplicada a la economía con R y Python
## Informe Investigativo: Modelado Econométrico Y Complementariedad Metodológica En El Empalme De Series Pib Del Dane (1994-2026)

---

**Institución:** Fundación Universitaria del Área Andina
**Materia:** Inteligencia Artificial para la solución de problemas

**Integrantes:**
1. Andrés Fabián Sepúlveda Marino
2. Angie Camila Velasquez
3. Stefania Colorado Tico
4. Jhony Stevan Cárdenas Rodríguez

 **Profesor:** Luz Andrea Sánchez Buitrago

**Ciudad:** Bogotá D.C., Colombia

**Fecha:** 31 de mayo de 2026  

---

In [ ]:
# ==============================================================================
# CELDA 1: Clonación del Repositorio, Instalación e Importación de Librerías
# ==============================================================================
# 1. Limpieza y clonación del repositorio
!rm -rf pib-colombia-arima
!git clone https://github.com/Disskaf/pib-colombia-arima.git

# 2. Instalación de librerías necesarias
!pip install pmdarima openpyxl xlrd msoffcrypto-tool

# 3. Adición de la ruta del clon al sistema de Python para poder importar el motor
import sys
sys.path.append('/content/pib-colombia-arima')

# 4. Importaciones generales
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from pmdarima import auto_arima
import warnings

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

In [ ]:
# ==============================================================================
# CELDA 2: Procesamiento ETL con Desencriptación, Extracción y Empalme Macroeconómico
# ==============================================================================
# Importar la función principal del motor de extracción recién clonado
from src.extraction_engine import procesar_y_empalmar

# Ruta donde se alojan los 65 archivos descargados por git clone
ruta_carpeta = '/content/pib-colombia-arima/data'

# Ejecución del pipeline ETL modularizado
df_pib, df_total_original, df_base2015, df_base2005, df_base1994 = procesar_y_empalmar(ruta_carpeta)

print("\nEstructura final de los datos reales empalmados (Primeros 5 registros):")
display(df_pib.head())

In [ ]:
# ==============================================================================
# CELDA 3: Validación Técnica, Control de Saltos y Consistencia Macroeconómica
# ==============================================================================
print("="*80)
print("SISTEMA DE CONTROL DE CALIDAD Y DIAGNÓSTICO DE LA SERIE EMPALMADA")
print("="*80)

# 1. Validación de Gaps (Continuidad Cronológica)
fecha_inicio, fecha_fin = df_pib.index.min(), df_pib.index.max()
rango_completo = pd.date_range(start=fecha_inicio, end=fecha_fin, freq='Q')
gaps = rango_completo.difference(df_pib.index)

print(f"Rango temporal total: Desde {fecha_inicio.strftime('%Y-%m')} hasta {fecha_fin.strftime('%Y-%m')}")
print(f"Observaciones totales esperadas: {len(rango_completo)}")
print(f"Observaciones reales consolidadas: {len(df_pib)}")

if len(gaps) == 0:
    print("✅ Continuidad temporal confirmada: No existen lagunas o trimestres faltantes en la serie.")
else:
    print(f"⚠️ Alerta de Gaps detectados: Faltan los siguientes trimestres: {gaps.tolist()}")

# 2. Cuantificación de los Saltos Metodológicos (Escala de Ajuste)
punto_empalme_2015 = df_base2015.index.min()
punto_empalme_2005 = df_base2005.index.min() if not df_base2005.empty else None

print("\n--- Análisis de Escala de Precios en los Puntos de Cruce ---")

if punto_empalme_2015 in df_base2005.index:
    const_2015 = df_base2015.loc[punto_empalme_2015]
    const_2005 = df_base2005.loc[punto_empalme_2015]
    for sector in ['Construccion', 'Sector_Financiero']:
        ratio = (const_2015[sector] / const_2005[sector])
        porcentaje_ajuste = (ratio - 1) * 100
        print(f"Sector {sector} (Base 2015 vs 2005 en {punto_empalme_2015.strftime('%Y-%m')}):")
        print(f"  - Valor Base 2005: {const_2005[sector]:,.2f}")
        print(f"  - Valor Base 2015: {const_2015[sector]:,.2f}")
        print(f"  - Ajuste de escala: {porcentaje_ajuste:+.2f}%")

if punto_empalme_2005 and punto_empalme_2005 in df_base1994.index:
    const_2005_p = df_base2005.loc[punto_empalme_2005]
    const_1994 = df_base1994.loc[punto_empalme_2005]
    for sector in ['Construccion', 'Sector_Financiero']:
        ratio = (const_2005_p[sector] / const_1994[sector])
        porcentaje_ajuste = (ratio - 1) * 100
        print(f"Sector {sector} (Base 2005 vs 1994 en {punto_empalme_2005.strftime('%Y-%m')}):")
        print(f"  - Valor Base 1994: {const_1994[sector]:,.2f}")
        print(f"  - Valor Base 2005: {const_2005_p[sector]:,.2f}")
        print(f"  - Ajuste de escala: {porcentaje_ajuste:+.2f}%")

# 3. Detección de Anomalías Estadísticas (Outliers)
print("\n--- Diagnóstico de Comportamientos Atípicos (Z-Scores > 3.0) ---")
for sector in ['Construccion', 'Sector_Financiero']:
    tasas_var = df_pib[sector].pct_change().dropna()
    z_scores = np.abs((tasas_var - tasas_var.mean()) / tasas_var.std())
    anomalas = z_scores[z_scores > 3.0]

    if len(anomalas) > 0:
        print(f"Sector {sector}: Se detectan {len(anomalas)} variaciones inusuales:")
        for idx, z_val in anomalas.items():
            variacion = tasas_var.loc[idx] * 100
            print(f"  * Periodo {idx.strftime('%Y-%m')}: Variación de {variacion:+.2f}% (Z-score: {z_val:.2f})")
    else:
        print(f"Sector {sector}: No se registran anomalías estadísticas fuera de los rangos normales.")

# 4. Gráfico de Diagnóstico del Empalme (Visualización con Doble Eje Y)
fig, ax = plt.subplots(2, 1, figsize=(14, 12))

# Subplot 1: Sector Financiero
ax[0].plot(df_pib.index, df_pib['Sector_Financiero'], label='Serie Empalmada Unificada (Izq - Escala 2015)', color='#1f77b4', linewidth=2.5)
if not df_base2015.empty:
    ax[0].plot(df_base2015.index, df_base2015['Sector_Financiero'], label='Base 2015 Original (Izq)', color='#1f77b4', linestyle='--', alpha=0.7)
if not df_base2005.empty:
    ax[0].plot(df_base2005.index, df_base2005['Sector_Financiero'], label='Base 2005 Original (Izq)', color='#d62728', linestyle=':', alpha=0.8)

ax[0].axvline(punto_empalme_2015, color='gray', linestyle='--', alpha=0.5, label='Empalme 2015')
if punto_empalme_2005:
    ax[0].axvline(punto_empalme_2005, color='black', linestyle='-.', alpha=0.5, label='Empalme 2005')
ax[0].set_title('Verificación del Empalme Metodológico: Sector Financiero', fontsize=12)
ax[0].set_ylabel('Escala Base 2015 / 2005 (Miles de Millones de Pesos)', color='#1f77b4')
ax[0].tick_params(axis='y', labelcolor='#1f77b4')

# Eje secundario para la Base 1994 (en millones de pesos)
if not df_base1994.empty:
    ax0_right = ax[0].twinx()
    ax0_right.plot(df_base1994.index, df_base1994['Sector_Financiero'], label='Base 1994 Original (Der - Escala 1994)', color='purple', linestyle=':', alpha=0.6)
    ax0_right.set_ylabel('Escala Base 1994 Original (Millones de Pesos)', color='purple')
    ax0_right.tick_params(axis='y', labelcolor='purple')
    ax0_right.grid(False)

    # Consolidación de leyendas
    lines, labels = ax[0].get_legend_handles_labels()
    lines2, labels2 = ax0_right.get_legend_handles_labels()
    ax[0].legend(lines + lines2, labels + labels2, loc='upper left')
else:
    ax[0].legend(loc='upper left')

# Subplot 2: Construcción
ax[1].plot(df_pib.index, df_pib['Construccion'], label='Serie Empalmada Unificada (Izq - Escala 2015)', color='#ff7f0e', linewidth=2.5)
if not df_base2015.empty:
    ax[1].plot(df_base2015.index, df_base2015['Construccion'], label='Base 2015 Original (Izq)', color='#ff7f0e', linestyle='--', alpha=0.7)
if not df_base2005.empty:
    ax[1].plot(df_base2005.index, df_base2005['Construccion'], label='Base 2005 Original (Izq)', color='#d62728', linestyle=':', alpha=0.8)

ax[1].axvline(punto_empalme_2015, color='gray', linestyle='--', alpha=0.5, label='Empalme 2015')
if punto_empalme_2005:
    ax[1].axvline(punto_empalme_2005, color='black', linestyle='-.', alpha=0.5, label='Empalme 2005')
ax[1].set_title('Verificación del Empalme Metodológico: Construcción', fontsize=12)
ax[1].set_ylabel('Escala Base 2015 / 2005 (Miles de Millones de Pesos)', color='#ff7f0e')
ax[1].tick_params(axis='y', labelcolor='#ff7f0e')

# Eje secundario para la Base 1994 (en millones de pesos)
if not df_base1994.empty:
    ax1_right = ax[1].twinx()
    ax1_right.plot(df_base1994.index, df_base1994['Construccion'], label='Base 1994 Original (Der - Escala 1994)', color='purple', linestyle=':', alpha=0.6)
    ax1_right.set_ylabel('Escala Base 1994 Original (Millones de Pesos)', color='purple')
    ax1_right.tick_params(axis='y', labelcolor='purple')
    ax1_right.grid(False)

    # Consolidación de leyendas
    lines, labels = ax[1].get_legend_handles_labels()
    lines2, labels2 = ax1_right.get_legend_handles_labels()
    ax[1].legend(lines + lines2, labels + labels2, loc='upper left')
else:
    ax[1].legend(loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# CELDA 4: Gráficas de Líneas (Comportamiento y Tendencia)
# ==============================================================================
plt.figure(figsize=(14, 6))

plt.plot(df_pib.index, df_pib['Sector_Financiero'], label='Sector Financiero', color='#1f77b4', linewidth=2)
plt.plot(df_pib.index, df_pib['Construccion'], label='Construcción', color='#ff7f0e', linewidth=2)

plt.title('Evolución del PIB a Precios Constantes (Histórico Real - Empalmado Base 2015)', fontsize=14, pad=15)
plt.xlabel('Año', fontsize=12)
plt.ylabel('Miles de Millones de Pesos', fontsize=12)
plt.legend(loc='upper left', fontsize=11)
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# CELDA 5: Análisis de Estacionariedad y Parámetros ARIMA (p, d, q)
# ==============================================================================
def analizar_estacionariedad_y_rezagos(serie, nombre_sector):
    """
    Aplica la prueba de Dickey-Fuller Aumentada y grafica ACF/PACF.
    """
    serie = serie.dropna()
    print(f"\n--- Análisis del Sector: {nombre_sector} ---")

    resultado_adf = adfuller(serie)
    print(f"Estadístico ADF: {resultado_adf[0]:.4f}")
    print(f"Valor p: {resultado_adf[1]:.4f}")

    if resultado_adf[1] > 0.05:
        print("Conclusión: La serie NO es estacionaria. Requiere diferenciación (d > 0).")
        serie_analisis = serie.diff().dropna()
        titulo_grafico = f"ACF y PACF - {nombre_sector} (Diferenciada, d=1)"
    else:
        print("Conclusión: La serie ES estacionaria (d = 0).")
        serie_analisis = serie
        titulo_grafico = f"ACF y PACF - {nombre_sector} (Original)"

    fig, ax = plt.subplots(1, 2, figsize=(14, 4))
    plot_acf(serie_analisis, ax=ax[0], title=f'ACF - Rezagos MA (q)')
    plot_pacf(serie_analisis, ax=ax[1], title=f'PACF - Rezagos AR (p)')
    fig.suptitle(titulo_grafico, fontsize=12)
    plt.tight_layout()
    plt.show()

analizar_estacionariedad_y_rezagos(df_pib['Sector_Financiero'], 'Sector Financiero')
analizar_estacionariedad_y_rezagos(df_pib['Construccion'], 'Construcción')

In [ ]:
# ==============================================================================
# CELDA 6: Modelado ARIMA y Pronóstico a 10 Períodos
# ==============================================================================
def proyectar_arima(serie, nombre_sector, periodos=10):
    """
    Busca el mejor modelo ARIMA automáticamente y proyecta 10 trimestres.
    """
    serie = serie.dropna()
    print(f"\nAjustando modelo ARIMA para {nombre_sector}...")

    modelo_optimo = auto_arima(serie,
                               seasonal=False,
                               stepwise=True,
                               suppress_warnings=True,
                               error_action="ignore",
                               trace=False)

    print(f"Parámetros óptimos seleccionados: {modelo_optimo.order} (p, d, q)")
    print(modelo_optimo.summary().tables[1])

    pronostico, conf_int = modelo_optimo.predict(n_periods=periodos, return_conf_int=True)

    ultima_fecha = serie.index[-1]
    fechas_futuras = pd.date_range(start=ultima_fecha, periods=periodos + 1, freq='Q')[1:]

    plt.figure(figsize=(12, 5))
    plt.plot(serie.index[-30:], serie.iloc[-30:], label='Histórico (Últimos 30 trimestres)', color='#2ca02c')
    plt.plot(fechas_futuras, pronostico, label='Pronóstico (10 periodos)', color='#d62728', linestyle='--')
    plt.fill_between(fechas_futuras, conf_int[:, 0], conf_int[:, 1], color='#d62728', alpha=0.2, label='Int. Confianza 95%')

    plt.title(f'Pronóstico ARIMA {modelo_optimo.order} - {nombre_sector}', fontsize=13)
    plt.xlabel('Año')
    plt.ylabel('Miles de Millones de Pesos')
    plt.legend(loc='upper left')
    plt.grid(True, alpha=0.5)
    plt.tight_layout()
    plt.show()

    return pronostico

pronostico_fin = proyectar_arima(df_pib['Sector_Financiero'], 'Sector Financiero')
pronostico_const = proyectar_arima(df_pib['Construccion'], 'Construcción')